# P1 v2 — BGE-M3 Embedding Fine-Tuning (Düzeltilmiş)
**Kaggle T4 x2 · Tek GPU · CENG493 Turkish Legal RAG**

### v1'den Farklar
| | v1 (hatalı) | v2 (düzeltilmiş) |
|---|---|---|
| Loss | `TripletLoss` | `MultipleNegativesRankingLoss` |
| MAX_SEQ_LEN | `128` (veri kesiliyordu) | `512` |
| MAX_TRAIN | `2000` (sample) | `1949` (tüm veri) |
| EPOCHS | `1` | `2` |
| BATCH_SIZE | `2` | `4` |
| Query filtresi | Yok | ≥3 kelime (gürültü temizlendi) |

### Neden MultipleNegativesRankingLoss?
- BGE-M3, MNRL ile pre-train edilmiş — aynı loss ile fine-tune etmek tutarlı
- Her batch'teki diğer tüm positivelar otomatik olarak in-batch negative görevi görür
- TripletLoss'ta sabit margin problemi yok
- Aynı batch size ile çok daha fazla negatif görür → daha iyi öğrenme

In [5]:
# ============================================================
# KRİTİK: torch import edilmeden ÖNCE — tek GPU görünür yap
# DataParallel → OOM'u önler
# ============================================================
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
print(f'Görünen GPU sayısı: {torch.cuda.device_count()}')  # 1 çıkmalı
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU 0: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory/1024**3:.1f} GB')
    print(f'Kullanılan: {torch.cuda.memory_allocated()/1024**3:.2f} GB')
else:
    print('GPU bulunamadı!')

Görünen GPU sayısı: 1
GPU 0: Tesla T4 | VRAM: 14.6 GB
Kullanılan: 0.00 GB


In [6]:
!pip install -q sentence-transformers==3.4.1 datasets tqdm

In [7]:
import os, json, random, gc
from pathlib import Path
from dataclasses import dataclass

@dataclass
class Config:
    # ── Kaggle Input Dataset klasör yolu — GEREKİRSE DEĞİŞTİR ──
    DRIVE_DIR    : str   = '/kaggle/input/datasets/ardayildiz29/legalo'
    EMBED_FILE   : str   = 'embedding.jsonl'

    # Model
    EMBED_BASE   : str   = 'BAAI/bge-m3'

    # ── EĞİTİM PARAMETRELERİ (v1'den farklar burada) ──
    MIN_QUERY_LEN : int  = 3        # Bu kelimenin altındaki query'ler atlanır
    MAX_TRAIN    : int   = 9999     # 9999 = tüm veri kullan
    BATCH_SIZE   : int   = 4        # seq_len=512 ile T4 14.6GB için güvenli
    EPOCHS       : int   = 2        # v1'de 1 epoch yetmiyordu
    LR           : float = 2e-5
    MAX_SEQ_LEN  : int   = 512      # v1'de 128 → veri kesiliyordu, şimdi 512
    WARMUP_RATIO : float = 0.1
    RANDOM_SEED  : int   = 42

    # Çıktı
    OUTPUT_DIR   : str   = '/kaggle/working/bge_m3_ft_v2'

CFG = Config()
Path(CFG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f'Model      : {CFG.EMBED_BASE}')
print(f'Veri       : {CFG.DRIVE_DIR}/{CFG.EMBED_FILE}')
print(f'Batch      : {CFG.BATCH_SIZE}')
print(f'Epochs     : {CFG.EPOCHS}')
print(f'Max seq len: {CFG.MAX_SEQ_LEN}')
print(f'Output     : {CFG.OUTPUT_DIR}')

Model      : BAAI/bge-m3
Veri       : /kaggle/input/datasets/ardayildiz29/legalo/embedding.jsonl
Batch      : 4
Epochs     : 2
Max seq len: 512
Output     : /kaggle/working/bge_m3_ft_v2


In [8]:
from tqdm import tqdm

embed_path = Path(CFG.DRIVE_DIR) / CFG.EMBED_FILE
assert embed_path.exists(), f'Dosya bulunamadı: {embed_path}'

raw = []
with open(embed_path, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            raw.append(json.loads(line))

print(f'Ham veri: {len(raw)} satır')
print(f'Örnek keys: {list(raw[0].keys())}')

Ham veri: 1949 satır
Örnek keys: ['id', 'query', 'positive_passage', 'negative_passage', 'positive_id', 'negative_id', 'positive_citation', 'negative_citation', 'negative_type', 'source', 'metadata', 'positive_resolved_id', 'negative_resolved_id', 'citation_label_is_display_only']


In [9]:
from sentence_transformers import InputExample

random.seed(CFG.RANDOM_SEED)

examples = []
skipped_empty   = 0
skipped_short_q = 0

for item in raw:
    q   = item.get('query', '').strip()
    pos = item.get('positive_passage', '').strip()
    neg = item.get('negative_passage', '').strip()

    # Boş alan kontrolü
    if not (q and pos and neg):
        skipped_empty += 1
        continue

    # Çok kısa query filtresi — 'Adaylar?' gibi tek kelimeler gürültü
    if len(q.split()) < CFG.MIN_QUERY_LEN:
        skipped_short_q += 1
        continue

    examples.append(InputExample(texts=[q, pos, neg]))

# MAX_TRAIN sınırı (9999 = hepsi)
if len(examples) > CFG.MAX_TRAIN:
    examples = random.sample(examples, CFG.MAX_TRAIN)

print(f'Toplam ham    : {len(raw)}')
print(f'Atlanan (boş) : {skipped_empty}')
print(f'Atlanan (kısa query <{CFG.MIN_QUERY_LEN} kelime): {skipped_short_q}')
print(f'Geçerli triplet: {len(examples)}')
print()
print(f'Örnek query   : {examples[0].texts[0][:80]}')
print(f'Örnek positive: {examples[0].texts[1][:80]}')
print(f'Örnek negative: {examples[0].texts[2][:80]}')

2026-05-30 15:54:57.610250: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780156497.804773      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780156497.857285      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780156498.333174      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780156498.333208      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780156498.333211      58 computation_placer.cc:177] computation placer alr

Toplam ham    : 1949
Atlanan (boş) : 0
Atlanan (kısa query <3 kelime): 586
Geçerli triplet: 1363

Örnek query   : Hukuk Genel Kurulu 2024/235 E., 2025/211 K. sayılı Yargıtay HGK kararında Usuli 
Örnek positive: Aynı ilke Yargıtay Hukuk Genel Kurulunun 24.09.2019 tarihli ve 2015/21-3903 Esas
Örnek negative: Hukuk Dairesi kararının da yok hükmünde olduğunu, bağlantılı olarak Asliye Hukuk


In [10]:
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader

print('BGE-M3 yükleniyor (GPU)...')
embed_model = SentenceTransformer(CFG.EMBED_BASE, device='cuda')
embed_model.max_seq_length = CFG.MAX_SEQ_LEN

# Gradient checkpointing — bellek tasarrufu
try:
    embed_model[0].auto_model.gradient_checkpointing_enable()
    print('Gradient checkpointing: ON')
except Exception as e:
    print(f'Gradient checkpointing atlandı: {e}')

print(f'Model cihazı  : {embed_model.device}')
print(f'Max seq length: {embed_model.max_seq_length}')
print(f'VRAM kullanım : {torch.cuda.memory_allocated()/1024**3:.2f} GB')

BGE-M3 yükleniyor (GPU)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Gradient checkpointing: ON
Model cihazı  : cuda:0
Max seq length: 512
VRAM kullanım : 2.12 GB


In [11]:
train_loader = DataLoader(
    examples,
    shuffle=True,
    batch_size=CFG.BATCH_SIZE,
    drop_last=True,
    num_workers=0,
)

# ── DÜZELTME: TripletLoss → MultipleNegativesRankingLoss ──
#
# Neden MNRL?
#   1. BGE-M3 MNRL ile pre-train edilmiş → aynı paradigma
#   2. Batch içindeki diğer positive'lar otomatik in-batch negative olur
#      → batch_size=4 ile aslında 3 negatif görüyor, TripletLoss'ta sadece 1
#   3. Margin hyperparameter yok → tuning gerekmez
loss_fn = losses.MultipleNegativesRankingLoss(embed_model)

total_steps  = len(train_loader) * CFG.EPOCHS
warmup_steps = int(total_steps * CFG.WARMUP_RATIO)

print(f'Toplam adım : {total_steps}')
print(f'Warmup adım : {warmup_steps}')
print(f'Loss        : MultipleNegativesRankingLoss')
print(f'LR          : {CFG.LR}')
print('-' * 50)
print('Eğitim başlıyor...')

embed_model.fit(
    train_objectives=[(train_loader, loss_fn)],
    epochs=CFG.EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': CFG.LR},
    show_progress_bar=True,
    output_path=CFG.OUTPUT_DIR,
    save_best_model=True,
    checkpoint_save_steps=0,
    use_amp=False,   # T4 bf16 desteklemiyor
)

print(f'\n✅ Eğitim tamamlandı!')

Toplam adım : 680
Warmup adım : 68
Loss        : MultipleNegativesRankingLoss
LR          : 2e-05
--------------------------------------------------
Eğitim başlıyor...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.221500



✅ Eğitim tamamlandı!


In [12]:
del embed_model
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM sonrası: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

print(f'\nKaydedilen dosyalar — {CFG.OUTPUT_DIR}:')
total = 0
for f in sorted(Path(CFG.OUTPUT_DIR).rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1024**2
        total  += size_mb
        print(f'  {str(f.relative_to(CFG.OUTPUT_DIR)):<45} {size_mb:>8.1f} MB')
print(f'  {"TOPLAM":<45} {total:>8.1f} MB')

VRAM sonrası: 2.13 GB

Kaydedilen dosyalar — /kaggle/working/bge_m3_ft_v2:
  1_Pooling/config.json                              0.0 MB
  README.md                                          0.0 MB
  config.json                                        0.0 MB
  config_sentence_transformers.json                  0.0 MB
  model.safetensors                               2165.9 MB
  modules.json                                       0.0 MB
  sentence_bert_config.json                          0.0 MB
  sentencepiece.bpe.model                            4.8 MB
  special_tokens_map.json                            0.0 MB
  tokenizer.json                                    16.3 MB
  tokenizer_config.json                              0.0 MB
  TOPLAM                                          2187.0 MB


In [13]:
# Hızlı doğrulama
print('Fine-tuned model test ediliyor...')
test_model = SentenceTransformer(CFG.OUTPUT_DIR, device='cpu')

sorgular = [
    'Susma hakkı nedir?',
    'Kiracının tahliye edilmesi nasıl gerçekleşir?',
    'İş sözleşmesinin feshi için geçerli sebepler nelerdir?',
]
embeddings = test_model.encode(sorgular, normalize_embeddings=True)
print(f'Embedding boyutu : {embeddings.shape}')  # (3, 1024) bekleniyor
print(f'İlk embed örneği : {embeddings[0][:5]}')

# Kendi içinde benzerlik kontrolü — ilk iki sorgu birbirinden farklı olmalı
import numpy as np
sim_01 = float(np.dot(embeddings[0], embeddings[1]))
sim_02 = float(np.dot(embeddings[0], embeddings[2]))
print(f'\nSorgu 1-2 benzerlik: {sim_01:.3f}')
print(f'Sorgu 1-3 benzerlik: {sim_02:.3f}')
print('(Normalize embeddings için -1..1 arası, yüksek = benzer)')
print('\n✅ Model başarıyla yüklendi ve çalışıyor!')

del test_model
gc.collect()

Fine-tuned model test ediliyor...


The tokenizer you are loading from '/kaggle/working/bge_m3_ft_v2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Embedding boyutu : (3, 1024)
İlk embed örneği : [-0.01818003  0.02188398  0.00936188 -0.0190928   0.01274086]

Sorgu 1-2 benzerlik: 0.046
Sorgu 1-3 benzerlik: 0.209
(Normalize embeddings için -1..1 arası, yüksek = benzer)

✅ Model başarıyla yüklendi ve çalışıyor!


7094

In [23]:
# Kaggle Dataset olarak yükle
# OUTPUT_DIR altındaki klasör adı 'bgem3' olsun — diğer notebook'lar bunu bekliyor
import shutil

FINAL_DIR = Path('/kaggle/working/bgem3')
if FINAL_DIR.exists():
    shutil.rmtree(FINAL_DIR)
shutil.copytree(CFG.OUTPUT_DIR, FINAL_DIR)
print(f'Model kopyalandı: {FINAL_DIR}')

# Dataset metadata
metadata = {
    "title": "Legal Models Tr Finetuned",
    "id": "ardayildiz29/legal-models-tr-finetuned",
    "licenses": [{"name": "CC0-1.0"}]
}
with open(FINAL_DIR / 'dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

print('Dataset oluşturuluyor...')
# Mevcut dataset'i güncelle (zaten varsa)
result = os.system(f'kaggle datasets version -p {FINAL_DIR} -m "v2: MNRL loss, seq512, 2epoch" --dir-mode zip')

if result != 0:
    # İlk kez oluşturuluyorsa
    result = os.system(f'kaggle datasets create -p {FINAL_DIR} --dir-mode zip')

if result == 0:
    print('✅ Dataset yüklendi!')
    print()
    print('Diğer notebook\'larda kullanmak için:')
    print('  MODEL_DIR = "/kaggle/input/datasets/ardayildiz29/legal-models-tr-finetuned"')
    print('  FT_EMBED_PATH = Path(MODEL_DIR) / "bgem3"')
else:
    print('❌ Otomatik yükleme başarısız.')
    print('Manuel yükleme: Kaggle > Output > bge_m3_ft_v2 klasörünü dataset olarak ekle')

Model kopyalandı: /kaggle/working/bgem3
Dataset oluşturuluyor...
Starting upload for file 2_Normalize.zip


100%|██████████| 22.0/22.0 [00:00<00:00, 121B/s]


Upload successful: 2_Normalize.zip (22B)
Starting upload for file 1_Pooling.zip


100%|██████████| 248/248 [00:00<00:00, 1.40kB/s]


Upload successful: 1_Pooling.zip (248B)
Starting upload for file tokenizer_config.json
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'


...

Upload successful: tokenizer.json (16MB)
Starting upload for file sentencepiece.bpe.model
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'


100%|██████████| 4.83M/4.83M [00:00<00:00, 21.3MB/s]
  0%|          | 0.00/964 [00:00<?, ?B/s]

Upload successful: sentencepiece.bpe.model (5MB)
Starting upload for file special_tokens_map.json
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'


100%|██████████| 964/964 [00:00<00:00, 5.18kB/s]


Upload successful: special_tokens_map.json (964B)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/ardayildiz29/legal-models-tr-finetuned
✅ Dataset yüklendi!

Diğer notebook'larda kullanmak için:
  MODEL_DIR = "/kaggle/input/datasets/ardayildiz29/legal-models-tr-finetuned"
  FT_EMBED_PATH = Path(MODEL_DIR) / "bgem3"


In [19]:
print('=' * 60)
print('P1 v2 Embedding FT TAMAMLANDI')
print('=' * 60)
print()
print('Yapılan değişiklikler:')
print('  ✅ TripletLoss → MultipleNegativesRankingLoss')
print('  ✅ MAX_SEQ_LEN: 128 → 512')
print('  ✅ EPOCHS: 1 → 2')
print('  ✅ BATCH_SIZE: 2 → 4')
print('  ✅ Kısa query filtresi (<3 kelime atlandı)')
print('  ✅ Tüm veri kullanıldı (sample yerine)')
print()
print('Sonraki adım:')
print('  S2, S4, S5, S6, S7, S8 notebook\'larında EMBED_BASE değiştir:')
print('  EMBED_BASE = "/kaggle/input/datasets/ardayildiz29/legal-models-tr-finetuned/bgem3"')

P1 v2 Embedding FT TAMAMLANDI

Yapılan değişiklikler:
  ✅ TripletLoss → MultipleNegativesRankingLoss
  ✅ MAX_SEQ_LEN: 128 → 512
  ✅ EPOCHS: 1 → 2
  ✅ BATCH_SIZE: 2 → 4
  ✅ Kısa query filtresi (<3 kelime atlandı)
  ✅ Tüm veri kullanıldı (sample yerine)

Sonraki adım:
  S2, S4, S5, S6, S7, S8 notebook'larında EMBED_BASE değiştir:
  EMBED_BASE = "/kaggle/input/datasets/ardayildiz29/tr-legal-models-finetuned/bgem3"
